# Standalone Watch SSL Pipeline

This notebook is fully self-contained: no repo clone and no local imports. Point it at your cached `artifacts/windows` folder in Google Drive and it can:

- optionally run `watch_full`,
- run `watch_10pct`,
- run phone-watch contrastive pretraining,
- run all six frozen probes,
- generate the final comparison summary.

If you already have an older full watch baseline named `standalone_supervised_watch`, keep the default alias in the config cell and the comparison report will reuse it as `watch_full`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
print('Drive mounted')

In [ ]:
WINDOWS_ROOT = '/content/drive/MyDrive/cs286-project/artifacts/windows'
OUTPUT_ROOT = '/content/drive/MyDrive/cs286-project-runs'
FORCE_FRESH_RUN = False

RUN_WATCH_FULL = False
RUN_WATCH_10PCT = True
RUN_CONTRASTIVE = True
RUN_PROBES = True
RUN_COMPARE = True

WATCH_FULL_RUN_NAME = 'watch-full'
WATCH_FULL_EXPERIMENT = 'watch_full'
WATCH_10PCT_RUN_NAME = 'watch-10pct'
WATCH_10PCT_EXPERIMENT = 'watch_10pct'

CONTRASTIVE_RUN_NAME = 'pretrain'
CONTRASTIVE_EXPERIMENT = 'phone_watch_contrastive'

PROBE_RUNS = [
    ('pair', 0.1, 'pair-probe-10pct', 'contrastive_pair_probe_10pct'),
    ('pair', 1.0, 'pair-probe-100pct', 'contrastive_pair_probe_100pct'),
    ('watch', 0.1, 'watch-probe-10pct', 'contrastive_watch_probe_10pct'),
    ('watch', 1.0, 'watch-probe-100pct', 'contrastive_watch_probe_100pct'),
    ('phone', 0.1, 'phone-probe-10pct', 'contrastive_phone_probe_10pct'),
    ('phone', 1.0, 'phone-probe-100pct', 'contrastive_phone_probe_100pct'),
]

EXPERIMENT_ALIASES = {
    'standalone_supervised_watch': 'watch_full',
}
EXPECTED_EXPERIMENTS = [
    'watch_full',
    'watch_10pct',
    'contrastive_pair_probe_10pct',
    'contrastive_pair_probe_100pct',
    'contrastive_watch_probe_10pct',
    'contrastive_watch_probe_100pct',
    'contrastive_phone_probe_10pct',
    'contrastive_phone_probe_100pct',
]

SEED = 42
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 40
PATIENCE = 7
GRAD_CLIP = 1.0
DROPOUT = 0.2
NUM_WORKERS = 2
SAVE_EVERY = 1

TEMPERATURE = 0.2
PROJECTION_HIDDEN_DIM = 128
PROJECTION_OUT_DIM = 64
JITTER_STD = 0.01
SCALE_MIN = 0.9
SCALE_MAX = 1.1
MASK_RATIO = 0.1
MASK_LENGTH = 6
AUGMENTATIONS_ENABLED = True

MAX_TRAIN_WINDOWS = None
MAX_VAL_WINDOWS = None
MAX_TEST_WINDOWS = None

print({
    'windows_root': WINDOWS_ROOT,
    'output_root': OUTPUT_ROOT,
    'run_watch_full': RUN_WATCH_FULL,
    'run_watch_10pct': RUN_WATCH_10PCT,
    'run_contrastive': RUN_CONTRASTIVE,
    'run_probes': RUN_PROBES,
    'run_compare': RUN_COMPARE,
    'force_fresh_run': FORCE_FRESH_RUN,
})

In [ ]:
import csv
import json
import math
import pickle
import random
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def require_cuda() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available. In Colab, switch the runtime to GPU.')
    device = torch.device('cuda')
    print('Using GPU:', torch.cuda.get_device_name(0))
    return device


@dataclass(frozen=True)
class CachedWindowSample:
    split: str
    raw_split: str
    subject_id: int
    activity_label: str
    occurrence_index: int
    offset_index: int
    center_s: float
    start_s: float
    end_s: float
    window_id: int
    label_idx: int
    x_fusion: np.ndarray
    x_phone: np.ndarray
    x_watch: np.ndarray


@dataclass(frozen=True)
class CachedSplitData:
    split: str
    samples: tuple[CachedWindowSample, ...]

    def __len__(self) -> int:
        return len(self.samples)


class WindowRepository:
    def __init__(self, windows_root: str | Path):
        self.windows_root = Path(windows_root)
        manifest_path = self.windows_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'Missing manifest: {manifest_path}')
        self.manifest = json.loads(manifest_path.read_text())

    def load_split(self, split: str) -> CachedSplitData:
        metadata_path = self.windows_root / split / 'metadata.csv'
        if not metadata_path.exists():
            raise FileNotFoundError(f'Missing metadata: {metadata_path}')
        rows = []
        with metadata_path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rows.append({
                    'split': row['split'],
                    'raw_split': row['raw_split'],
                    'subject_id': int(row['subject_id']),
                    'activity_label': row['activity_label'],
                    'occurrence_index': int(row['occurrence_index']),
                    'offset_index': int(row['offset_index']),
                    'center_s': float(row['center_s']),
                    'start_s': float(row['start_s']),
                    'end_s': float(row['end_s']),
                    'window_id': int(row['window_id']),
                    'label_idx': int(row['label_idx']),
                })
        payloads = {}
        for chunk_path in sorted((self.windows_root / split).glob('data_chunk_*.pkl')):
            with chunk_path.open('rb') as handle:
                records = pickle.load(handle)
            for record in records:
                payloads[int(record['window_id'])] = np.asarray(record['x_fusion'], dtype=np.float32)
        samples = []
        for row in rows:
            payload = payloads[row['window_id']]
            x_phone = payload[0:6].copy()
            x_watch = payload[6:12].copy()
            samples.append(CachedWindowSample(
                split=row['split'],
                raw_split=row['raw_split'],
                subject_id=row['subject_id'],
                activity_label=row['activity_label'],
                occurrence_index=row['occurrence_index'],
                offset_index=row['offset_index'],
                center_s=row['center_s'],
                start_s=row['start_s'],
                end_s=row['end_s'],
                window_id=row['window_id'],
                label_idx=row['label_idx'],
                x_fusion=payload.copy(),
                x_phone=x_phone,
                x_watch=x_watch,
            ))
        return CachedSplitData(split=split, samples=tuple(samples))


def limit_samples(samples, max_samples):
    return samples if max_samples is None else tuple(samples[:max_samples])


def select_labeled_subset(samples, label_fraction: float, seed: int):
    if label_fraction >= 1.0:
        return tuple(samples)
    if label_fraction <= 0.0:
        raise ValueError('label_fraction must be > 0')
    rng = random.Random(seed)
    by_subject = {}
    for sample in samples:
        by_subject.setdefault(sample.subject_id, []).append(sample)
    selected = []
    for subject_id in sorted(by_subject):
        subject_samples = list(by_subject[subject_id])
        rng.shuffle(subject_samples)
        keep = max(1, int(round(len(subject_samples) * label_fraction)))
        selected.extend(subject_samples[:keep])
    selected.sort(key=lambda sample: (sample.subject_id, sample.window_id))
    return tuple(selected)


class TimeSeriesEncoder(nn.Module):
    output_dim = 256

    def __init__(self, in_channels: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x).squeeze(-1)


class SupervisedHARModel(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.encoder = TimeSeriesEncoder(in_channels=in_channels)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.encoder.output_dim, num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.encoder(x))


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int = TimeSeriesEncoder.output_dim, hidden_dim: int = 128, out_dim: int = 64):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.layers(x), dim=1)


class PhoneWatchContrastiveModel(nn.Module):
    def __init__(self, in_channels: int = 6, projection_hidden_dim: int = 128, projection_out_dim: int = 64):
        super().__init__()
        self.phone_encoder = TimeSeriesEncoder(in_channels=in_channels)
        self.watch_encoder = TimeSeriesEncoder(in_channels=in_channels)
        self.phone_projector = ProjectionHead(self.phone_encoder.output_dim, projection_hidden_dim, projection_out_dim)
        self.watch_projector = ProjectionHead(self.watch_encoder.output_dim, projection_hidden_dim, projection_out_dim)

    def encode_phone(self, x_phone: torch.Tensor) -> torch.Tensor:
        return self.phone_encoder(x_phone)

    def encode_watch(self, x_watch: torch.Tensor) -> torch.Tensor:
        return self.watch_encoder(x_watch)

    def forward(self, x_phone: torch.Tensor, x_watch: torch.Tensor):
        h_phone = self.encode_phone(x_phone)
        h_watch = self.encode_watch(x_watch)
        z_phone = self.phone_projector(h_phone)
        z_watch = self.watch_projector(h_watch)
        return h_phone, h_watch, z_phone, z_watch


class LinearProbeHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


def symmetric_info_nce_loss(z_phone: torch.Tensor, z_watch: torch.Tensor, temperature: float = 0.2):
    z_phone = F.normalize(z_phone, dim=1)
    z_watch = F.normalize(z_watch, dim=1)
    similarity = z_phone @ z_watch.T / temperature
    targets = torch.arange(z_phone.shape[0], device=z_phone.device)
    p2w = F.cross_entropy(similarity, targets)
    w2p = F.cross_entropy(similarity.T, targets)
    return 0.5 * (p2w + w2p), p2w, w2p


def apply_default_augmentations(x: torch.Tensor, jitter_std=0.01, scale_min=0.9, scale_max=1.1, mask_ratio=0.1, mask_length=6, enabled=True):
    if not enabled:
        return x
    out = x.clone()
    if jitter_std > 0:
        out = out + torch.randn_like(out) * jitter_std
    if scale_min != 1.0 or scale_max != 1.0:
        scales = torch.empty((out.shape[0], 1, 1), device=out.device).uniform_(scale_min, scale_max)
        out = out * scales
    if mask_ratio > 0 and mask_length > 0:
        seq_len = out.shape[-1]
        num_masks = max(1, int(round(seq_len * mask_ratio / max(1, mask_length))))
        for batch_index in range(out.shape[0]):
            for _ in range(num_masks):
                start = int(torch.randint(0, max(1, seq_len - mask_length + 1), (1,), device=out.device).item())
                out[batch_index, :, start:start + mask_length] = 0.0
    return out


def confusion_matrix_from_predictions(y_true, y_pred, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for truth, pred in zip(y_true, y_pred):
        matrix[int(truth), int(pred)] += 1
    return matrix


def macro_f1_from_confusion(matrix, index_to_label):
    per_class = {}
    scores = []
    for class_index in range(matrix.shape[0]):
        tp = float(matrix[class_index, class_index])
        fp = float(matrix[:, class_index].sum() - tp)
        fn = float(matrix[class_index, :].sum() - tp)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[index_to_label[class_index]] = f1
        scores.append(f1)
    return float(sum(scores) / len(scores)), per_class


def per_subject_accuracy(y_true, y_pred, subject_ids):
    counts = {}
    for subject_id, truth, pred in zip(subject_ids, y_true, y_pred):
        key = int(subject_id)
        counts.setdefault(key, [0, 0])
        counts[key][0] += int(int(truth) == int(pred))
        counts[key][1] += 1
    return {subject_id: correct / total for subject_id, (correct, total) in sorted(counts.items())}


def compute_classification_metrics(y_true, y_pred, subject_ids, index_to_label, loss=None):
    matrix = confusion_matrix_from_predictions(y_true, y_pred, num_classes=len(index_to_label))
    accuracy = float(matrix.diagonal().sum() / max(1, matrix.sum()))
    macro_f1, per_class_f1 = macro_f1_from_confusion(matrix, index_to_label)
    return {
        'loss': loss,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'per_class_f1': per_class_f1,
        'per_subject_accuracy': per_subject_accuracy(y_true, y_pred, subject_ids),
        'confusion_matrix': matrix,
    }


@dataclass
class EarlyStopper:
    patience: int
    mode: str = 'max'
    min_delta: float = 0.0
    best_score: float | None = None
    best_epoch: int | None = None
    epochs_without_improvement: int = 0
    should_stop: bool = False
    history: list[dict] = field(default_factory=list)

    def update(self, score: float, epoch: int) -> bool:
        improved = self.best_score is None or (score > self.best_score + self.min_delta if self.mode == 'max' else score < self.best_score - self.min_delta)
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.epochs_without_improvement = 0
            self.should_stop = False
        else:
            self.epochs_without_improvement += 1
            self.should_stop = self.epochs_without_improvement >= self.patience
        self.history.append({'epoch': epoch, 'score': score, 'improved': improved, 'should_stop': self.should_stop})
        return improved

    def state_dict(self):
        return {
            'patience': self.patience,
            'mode': self.mode,
            'min_delta': self.min_delta,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'epochs_without_improvement': self.epochs_without_improvement,
            'should_stop': self.should_stop,
            'history': list(self.history),
        }

    @classmethod
    def from_state_dict(cls, state):
        stopper = cls(patience=int(state['patience']), mode=state['mode'], min_delta=float(state['min_delta']))
        stopper.best_score = state['best_score']
        stopper.best_epoch = state['best_epoch']
        stopper.epochs_without_improvement = int(state['epochs_without_improvement'])
        stopper.should_stop = bool(state['should_stop'])
        stopper.history = [dict(entry) for entry in state.get('history', [])]
        return stopper


def checkpoint_metadata(experiment_name, stage, config, manifest_path, extra=None):
    payload = {
        'experiment_name': experiment_name,
        'stage': stage,
        'config': dict(config),
        'manifest_path': str(manifest_path),
        'saved_at_utc': datetime.now(UTC).isoformat(),
    }
    if extra:
        payload.update(dict(extra))
    return payload


def save_checkpoint(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(dict(payload), path)
    return path


def load_checkpoint(path):
    return torch.load(Path(path), map_location='cpu', weights_only=False)


def metric_token(value: float) -> str:
    return f'{value:.4f}'.replace('.', 'p')


def build_supervised_arrays(split_data, channel_mode):
    if channel_mode == 'fusion':
        x = np.stack([sample.x_fusion for sample in split_data.samples]).astype(np.float32)
    elif channel_mode == 'phone':
        x = np.stack([sample.x_phone for sample in split_data.samples]).astype(np.float32)
    elif channel_mode == 'watch':
        x = np.stack([sample.x_watch for sample in split_data.samples]).astype(np.float32)
    else:
        raise ValueError(f'Unsupported channel_mode: {channel_mode}')
    y = np.asarray([sample.label_idx for sample in split_data.samples], dtype=np.int64)
    subjects = np.asarray([sample.subject_id for sample in split_data.samples], dtype=np.int64)
    return x, y, subjects


def build_contrastive_arrays(split_data):
    x_phone = np.stack([sample.x_phone for sample in split_data.samples]).astype(np.float32)
    x_watch = np.stack([sample.x_watch for sample in split_data.samples]).astype(np.float32)
    y = np.asarray([sample.label_idx for sample in split_data.samples], dtype=np.int64)
    subjects = np.asarray([sample.subject_id for sample in split_data.samples], dtype=np.int64)
    return x_phone, x_watch, y, subjects


def to_loader(*arrays, batch_size, shuffle, num_workers):
    tensors = [torch.from_numpy(array) for array in arrays]
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)


def plot_confusion_matrix(matrix, index_to_label, title, output_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap='Blues')
    labels = [index_to_label[index] for index in range(len(index_to_label))]
    ticks = np.arange(len(labels))
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_yticks(ticks)
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def write_subject_accuracy(path, per_subject_accuracy_map):
    with Path(path).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['subject_id', 'accuracy'])
        writer.writeheader()
        for subject_id, accuracy in per_subject_accuracy_map.items():
            writer.writerow({'subject_id': subject_id, 'accuracy': accuracy})


def evaluate_classifier(model, loader, subjects, loss_fn, index_to_label, device):
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_true, all_pred = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            total_loss += float(loss.item()) * len(x_batch)
            total_count += len(x_batch)
            all_true.extend(y_batch.cpu().tolist())
            all_pred.extend(logits.argmax(dim=1).cpu().tolist())
    return compute_classification_metrics(all_true, all_pred, subjects.tolist(), index_to_label, loss=total_loss / max(1, total_count))


def extract_probe_features(model, split_data, evaluation_mode, batch_size, device, num_workers):
    phone = np.stack([sample.x_phone for sample in split_data.samples]).astype(np.float32)
    watch = np.stack([sample.x_watch for sample in split_data.samples]).astype(np.float32)
    labels = np.asarray([sample.label_idx for sample in split_data.samples], dtype=np.int64)
    subjects = np.asarray([sample.subject_id for sample in split_data.samples], dtype=np.int64)
    loader = to_loader(phone, watch, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    features = []
    model.eval()
    with torch.no_grad():
        for x_phone, x_watch in loader:
            x_phone = x_phone.to(device)
            x_watch = x_watch.to(device)
            h_phone = model.encode_phone(x_phone)
            h_watch = model.encode_watch(x_watch)
            if evaluation_mode == 'pair':
                batch_features = torch.cat([h_phone, h_watch], dim=1)
            elif evaluation_mode == 'phone':
                batch_features = h_phone
            else:
                batch_features = h_watch
            features.append(batch_features.cpu().numpy())
    x = np.concatenate(features, axis=0)
    return x.astype(np.float32), labels, subjects


def freeze_module(module):
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)


def load_contrastive_model_from_checkpoint(checkpoint, device):
    config = checkpoint.get('metadata', {}).get('config', {})
    model = PhoneWatchContrastiveModel(
        in_channels=6,
        projection_hidden_dim=int(config.get('projection_hidden_dim', 128)),
        projection_out_dim=int(config.get('projection_out_dim', 64)),
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    freeze_module(model.phone_encoder)
    freeze_module(model.watch_encoder)
    freeze_module(model.phone_projector)
    freeze_module(model.watch_projector)
    return model


print('Standalone training helpers loaded')


def resolve_output_dir(stage: str, run_name: str) -> Path:
    return Path(OUTPUT_ROOT) / stage / run_name



def run_supervised_experiment(*, run_name: str, experiment_name: str, channel_mode: str, label_fraction: float) -> dict:
    seed_everything(SEED)
    device = require_cuda()
    windows_root = Path(WINDOWS_ROOT)
    output_dir = resolve_output_dir('supervised', run_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path = output_dir / 'latest_checkpoint.pt'

    repository = WindowRepository(windows_root)
    label_to_index = repository.manifest['label_to_index']
    index_to_label = {index: label for label, index in label_to_index.items()}

    train_split = repository.load_split('train')
    val_split = repository.load_split('val')
    test_split = repository.load_split('test')
    train_split = CachedSplitData('train', select_labeled_subset(train_split.samples, label_fraction, SEED))
    train_split = CachedSplitData('train', tuple(limit_samples(train_split.samples, MAX_TRAIN_WINDOWS)))
    val_split = CachedSplitData('val', tuple(limit_samples(val_split.samples, MAX_VAL_WINDOWS)))
    test_split = CachedSplitData('test', tuple(limit_samples(test_split.samples, MAX_TEST_WINDOWS)))

    train_x, train_y, train_subjects = build_supervised_arrays(train_split, channel_mode)
    val_x, val_y, val_subjects = build_supervised_arrays(val_split, channel_mode)
    test_x, test_y, test_subjects = build_supervised_arrays(test_split, channel_mode)

    model = SupervisedHARModel(train_x.shape[1], len(label_to_index), dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = to_loader(train_x, train_y, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = to_loader(val_x, val_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = to_loader(test_x, test_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    early_stopper = EarlyStopper(PATIENCE, mode='max')
    best_state = None
    history = []
    start_epoch = 1
    if latest_checkpoint_path.exists() and not FORCE_FRESH_RUN:
        resume = load_checkpoint(latest_checkpoint_path)
        model.load_state_dict(resume['model_state_dict'])
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        early_stopper = EarlyStopper.from_state_dict(resume['early_stopper_state_dict'])
        best_state = resume.get('best_model_state_dict')
        history = [dict(entry) for entry in resume.get('history', [])]
        start_epoch = int(resume['epoch']) + 1
        print(f'[{experiment_name}] resuming from epoch {start_epoch}')

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(x_batch)
            train_count += len(x_batch)
        train_loss = train_loss_sum / max(1, train_count)
        val_metrics = evaluate_classifier(model, val_loader, val_subjects, loss_fn, index_to_label, device)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_metrics['loss'], 'val_accuracy': val_metrics['accuracy'], 'val_macro_f1': val_metrics['macro_f1']})
        print(f'[{experiment_name}] epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_metrics["loss"]:.4f} val_acc={val_metrics["accuracy"]:.4f} val_macro_f1={val_metrics["macro_f1"]:.4f}')
        if early_stopper.update(float(val_metrics['macro_f1']), epoch):
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if SAVE_EVERY > 0 and (epoch % SAVE_EVERY == 0 or epoch == EPOCHS):
            save_checkpoint(latest_checkpoint_path, {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_model_state_dict': best_state,
                'early_stopper_state_dict': early_stopper.state_dict(),
                'history': history,
                'label_to_index': label_to_index,
                'metadata': checkpoint_metadata(experiment_name, 'supervised', {'channel_mode': channel_mode, 'label_fraction': label_fraction, 'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'train_windows_used': len(train_split), 'val_windows': len(val_split), 'test_windows': len(test_split), 'save_every': SAVE_EVERY, 'seed': SEED}, windows_root / 'manifest.json', {'checkpoint_role': 'latest', 'checkpoint_epoch': epoch}),
            })
        if early_stopper.should_stop:
            break

    if best_state is None:
        raise RuntimeError(f'[{experiment_name}] training finished without a best checkpoint')

    model.load_state_dict(best_state)
    train_metrics = evaluate_classifier(model, train_loader, train_subjects, loss_fn, index_to_label, device)
    val_metrics = evaluate_classifier(model, val_loader, val_subjects, loss_fn, index_to_label, device)
    test_metrics = evaluate_classifier(model, test_loader, test_subjects, loss_fn, index_to_label, device)

    stem = f'supervised_{channel_mode}_{experiment_name}_acc{metric_token(float(test_metrics["accuracy"]))}_macrof1{metric_token(float(test_metrics["macro_f1"]))}'
    checkpoint_path = output_dir / f'{stem}_checkpoint.pt'
    metrics_path = output_dir / f'{stem}_metrics.json'
    confusion_path = output_dir / f'{stem}_confusion_matrix.png'
    per_subject_path = output_dir / f'{stem}_per_subject_accuracy.csv'

    save_checkpoint(checkpoint_path, {'model_state_dict': model.state_dict(), 'label_to_index': label_to_index, 'best_epoch': early_stopper.best_epoch, 'metadata': checkpoint_metadata(experiment_name, 'supervised', {'channel_mode': channel_mode, 'label_fraction': label_fraction, 'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'train_windows_used': len(train_split), 'val_windows': len(val_split), 'test_windows': len(test_split), 'save_every': SAVE_EVERY, 'seed': SEED}, windows_root / 'manifest.json')})
    metrics_payload = {'experiment_name': experiment_name, 'best_epoch': early_stopper.best_epoch, 'channel_mode': channel_mode, 'label_fraction': label_fraction, 'train': {'loss': train_metrics['loss'], 'accuracy': train_metrics['accuracy'], 'macro_f1': train_metrics['macro_f1']}, 'val': {'loss': val_metrics['loss'], 'accuracy': val_metrics['accuracy'], 'macro_f1': val_metrics['macro_f1']}, 'test': {'loss': test_metrics['loss'], 'accuracy': test_metrics['accuracy'], 'macro_f1': test_metrics['macro_f1'], 'per_class_f1': test_metrics['per_class_f1'], 'per_subject_accuracy': test_metrics['per_subject_accuracy']}, 'history': history, 'artifacts': {'checkpoint_path': str(checkpoint_path), 'latest_checkpoint_path': str(latest_checkpoint_path), 'metrics_path': str(metrics_path), 'confusion_path': str(confusion_path), 'per_subject_accuracy_path': str(per_subject_path)}}
    metrics_path.write_text(json.dumps(metrics_payload, indent=2))
    plot_confusion_matrix(test_metrics['confusion_matrix'], index_to_label, experiment_name, confusion_path)
    write_subject_accuracy(per_subject_path, test_metrics['per_subject_accuracy'])
    return metrics_payload



def run_contrastive_experiment(*, run_name: str, experiment_name: str) -> dict:
    seed_everything(SEED)
    device = require_cuda()
    windows_root = Path(WINDOWS_ROOT)
    output_dir = resolve_output_dir('contrastive', run_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path = output_dir / 'latest_checkpoint.pt'

    repository = WindowRepository(windows_root)
    label_to_index = repository.manifest['label_to_index']
    train_split = CachedSplitData('train', tuple(limit_samples(repository.load_split('train').samples, MAX_TRAIN_WINDOWS)))
    val_split = CachedSplitData('val', tuple(limit_samples(repository.load_split('val').samples, MAX_VAL_WINDOWS)))
    train_phone, train_watch, _, _ = build_contrastive_arrays(train_split)
    val_phone, val_watch, _, _ = build_contrastive_arrays(val_split)

    model = PhoneWatchContrastiveModel(6, PROJECTION_HIDDEN_DIM, PROJECTION_OUT_DIM).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_loader = to_loader(train_phone, train_watch, np.zeros(len(train_phone), dtype=np.int64), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = to_loader(val_phone, val_watch, np.zeros(len(val_phone), dtype=np.int64), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    early_stopper = EarlyStopper(PATIENCE, mode='min')
    best_state = None
    history = []
    start_epoch = 1
    if latest_checkpoint_path.exists() and not FORCE_FRESH_RUN:
        resume = load_checkpoint(latest_checkpoint_path)
        model.load_state_dict(resume['model_state_dict'])
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        early_stopper = EarlyStopper.from_state_dict(resume['early_stopper_state_dict'])
        best_state = resume.get('best_model_state_dict')
        history = [dict(entry) for entry in resume.get('history', [])]
        start_epoch = int(resume['epoch']) + 1
        print(f'[{experiment_name}] resuming from epoch {start_epoch}')

    def eval_contrastive(loader):
        model.eval()
        total_loss = total_p2w = total_w2p = 0.0
        total_count = 0
        with torch.no_grad():
            for x_phone, x_watch, _ in loader:
                x_phone = x_phone.to(device)
                x_watch = x_watch.to(device)
                _, _, z_phone, z_watch = model(x_phone, x_watch)
                loss, p2w, w2p = symmetric_info_nce_loss(z_phone, z_watch, temperature=TEMPERATURE)
                batch_size = len(x_phone)
                total_loss += float(loss.item()) * batch_size
                total_p2w += float(p2w.item()) * batch_size
                total_w2p += float(w2p.item()) * batch_size
                total_count += batch_size
        return {'loss': total_loss / max(1, total_count), 'phone_to_watch_loss': total_p2w / max(1, total_count), 'watch_to_phone_loss': total_w2p / max(1, total_count)}

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for x_phone, x_watch, _ in train_loader:
            x_phone = apply_default_augmentations(x_phone.to(device), JITTER_STD, SCALE_MIN, SCALE_MAX, MASK_RATIO, MASK_LENGTH, AUGMENTATIONS_ENABLED)
            x_watch = apply_default_augmentations(x_watch.to(device), JITTER_STD, SCALE_MIN, SCALE_MAX, MASK_RATIO, MASK_LENGTH, AUGMENTATIONS_ENABLED)
            optimizer.zero_grad()
            _, _, z_phone, z_watch = model(x_phone, x_watch)
            loss, _, _ = symmetric_info_nce_loss(z_phone, z_watch, temperature=TEMPERATURE)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(x_phone)
            train_count += len(x_phone)
        train_loss = train_loss_sum / max(1, train_count)
        val_metrics = eval_contrastive(val_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_metrics['loss'], 'val_phone_to_watch_loss': val_metrics['phone_to_watch_loss'], 'val_watch_to_phone_loss': val_metrics['watch_to_phone_loss']})
        print(f'[{experiment_name}] epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_metrics["loss"]:.4f} val_p2w={val_metrics["phone_to_watch_loss"]:.4f} val_w2p={val_metrics["watch_to_phone_loss"]:.4f}')
        if early_stopper.update(float(val_metrics['loss']), epoch):
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if SAVE_EVERY > 0 and (epoch % SAVE_EVERY == 0 or epoch == EPOCHS):
            save_checkpoint(latest_checkpoint_path, {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_model_state_dict': best_state, 'early_stopper_state_dict': early_stopper.state_dict(), 'history': history, 'label_to_index': label_to_index, 'metadata': checkpoint_metadata(experiment_name, 'contrastive', {'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'temperature': TEMPERATURE, 'projection_hidden_dim': PROJECTION_HIDDEN_DIM, 'projection_out_dim': PROJECTION_OUT_DIM, 'jitter_std': JITTER_STD, 'scale_min': SCALE_MIN, 'scale_max': SCALE_MAX, 'mask_ratio': MASK_RATIO, 'mask_length': MASK_LENGTH, 'augmentations_enabled': AUGMENTATIONS_ENABLED, 'train_windows': len(train_split), 'val_windows': len(val_split), 'save_every': SAVE_EVERY, 'seed': SEED}, windows_root / 'manifest.json', {'checkpoint_role': 'latest', 'checkpoint_epoch': epoch})})
        if early_stopper.should_stop:
            break

    if best_state is None:
        raise RuntimeError(f'[{experiment_name}] contrastive training finished without a best checkpoint')

    last_epoch = int(history[-1]['epoch'])
    last_train_metrics = eval_contrastive(train_loader)
    last_val_metrics = eval_contrastive(val_loader)
    last_checkpoint_path = output_dir / f'contrastive_{experiment_name}_last_epoch{last_epoch:02d}_checkpoint.pt'
    save_checkpoint(last_checkpoint_path, {'model_state_dict': model.state_dict(), 'phone_encoder_state_dict': model.phone_encoder.state_dict(), 'watch_encoder_state_dict': model.watch_encoder.state_dict(), 'phone_projector_state_dict': model.phone_projector.state_dict(), 'watch_projector_state_dict': model.watch_projector.state_dict(), 'label_to_index': label_to_index, 'best_epoch': early_stopper.best_epoch, 'metadata': checkpoint_metadata(experiment_name, 'contrastive', {'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'temperature': TEMPERATURE, 'projection_hidden_dim': PROJECTION_HIDDEN_DIM, 'projection_out_dim': PROJECTION_OUT_DIM, 'jitter_std': JITTER_STD, 'scale_min': SCALE_MIN, 'scale_max': SCALE_MAX, 'mask_ratio': MASK_RATIO, 'mask_length': MASK_LENGTH, 'augmentations_enabled': AUGMENTATIONS_ENABLED, 'train_windows': len(train_split), 'val_windows': len(val_split), 'save_every': SAVE_EVERY, 'seed': SEED}, windows_root / 'manifest.json', {'checkpoint_role': 'last', 'checkpoint_epoch': last_epoch})})

    model.load_state_dict(best_state)
    best_train_metrics = eval_contrastive(train_loader)
    best_val_metrics = eval_contrastive(val_loader)
    stem = f'contrastive_{experiment_name}_valloss{best_val_metrics["loss"]:.4f}'.replace('.', 'p')
    checkpoint_path = output_dir / f'{stem}_checkpoint.pt'
    metrics_path = output_dir / f'{stem}_metrics.json'
    save_checkpoint(checkpoint_path, {'model_state_dict': model.state_dict(), 'phone_encoder_state_dict': model.phone_encoder.state_dict(), 'watch_encoder_state_dict': model.watch_encoder.state_dict(), 'phone_projector_state_dict': model.phone_projector.state_dict(), 'watch_projector_state_dict': model.watch_projector.state_dict(), 'label_to_index': label_to_index, 'best_epoch': early_stopper.best_epoch, 'metadata': checkpoint_metadata(experiment_name, 'contrastive', {'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'temperature': TEMPERATURE, 'projection_hidden_dim': PROJECTION_HIDDEN_DIM, 'projection_out_dim': PROJECTION_OUT_DIM, 'jitter_std': JITTER_STD, 'scale_min': SCALE_MIN, 'scale_max': SCALE_MAX, 'mask_ratio': MASK_RATIO, 'mask_length': MASK_LENGTH, 'augmentations_enabled': AUGMENTATIONS_ENABLED, 'train_windows': len(train_split), 'val_windows': len(val_split), 'save_every': SAVE_EVERY, 'seed': SEED}, windows_root / 'manifest.json', {'checkpoint_role': 'best', 'checkpoint_epoch': early_stopper.best_epoch})})
    metrics_payload = {'experiment_name': experiment_name, 'best_epoch': early_stopper.best_epoch, 'last_epoch': last_epoch, 'selection_metric': 'val_loss', 'train': best_train_metrics, 'val': best_val_metrics, 'last_train': last_train_metrics, 'last_val': last_val_metrics, 'history': history, 'artifacts': {'checkpoint_path': str(checkpoint_path), 'last_checkpoint_path': str(last_checkpoint_path), 'latest_checkpoint_path': str(latest_checkpoint_path), 'metrics_path': str(metrics_path)}}
    metrics_path.write_text(json.dumps(metrics_payload, indent=2))
    return metrics_payload



def run_probe_experiment(*, run_name: str, experiment_name: str, evaluation_mode: str, label_fraction: float, encoder_checkpoint_path: str) -> dict:
    seed_everything(SEED)
    device = require_cuda()
    windows_root = Path(WINDOWS_ROOT)
    output_dir = resolve_output_dir('probe', run_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path = output_dir / 'latest_checkpoint.pt'

    repository = WindowRepository(windows_root)
    label_to_index = repository.manifest['label_to_index']
    index_to_label = {index: label for label, index in label_to_index.items()}

    contrastive_checkpoint = load_checkpoint(encoder_checkpoint_path)
    encoder_model = load_contrastive_model_from_checkpoint(contrastive_checkpoint, device)
    train_split = repository.load_split('train')
    train_split = CachedSplitData('train', select_labeled_subset(train_split.samples, label_fraction, SEED))
    val_split = repository.load_split('val')
    test_split = repository.load_split('test')
    train_split = CachedSplitData('train', tuple(limit_samples(train_split.samples, MAX_TRAIN_WINDOWS)))
    val_split = CachedSplitData('val', tuple(limit_samples(val_split.samples, MAX_VAL_WINDOWS)))
    test_split = CachedSplitData('test', tuple(limit_samples(test_split.samples, MAX_TEST_WINDOWS)))

    train_x, train_y, train_subjects = extract_probe_features(encoder_model, train_split, evaluation_mode, BATCH_SIZE, device, NUM_WORKERS)
    val_x, val_y, val_subjects = extract_probe_features(encoder_model, val_split, evaluation_mode, BATCH_SIZE, device, NUM_WORKERS)
    test_x, test_y, test_subjects = extract_probe_features(encoder_model, test_split, evaluation_mode, BATCH_SIZE, device, NUM_WORKERS)

    head = LinearProbeHead(train_x.shape[1], len(label_to_index)).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = to_loader(train_x, train_y, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = to_loader(val_x, val_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = to_loader(test_x, test_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    early_stopper = EarlyStopper(PATIENCE, mode='max')
    best_state = None
    history = []
    start_epoch = 1
    if latest_checkpoint_path.exists() and not FORCE_FRESH_RUN:
        resume = load_checkpoint(latest_checkpoint_path)
        head.load_state_dict(resume['linear_head_state_dict'])
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        early_stopper = EarlyStopper.from_state_dict(resume['early_stopper_state_dict'])
        best_state = resume.get('best_linear_head_state_dict')
        history = [dict(entry) for entry in resume.get('history', [])]
        start_epoch = int(resume['epoch']) + 1
        print(f'[{experiment_name}] resuming from epoch {start_epoch}')

    def eval_probe(loader, subjects):
        return evaluate_classifier(head, loader, subjects, loss_fn, index_to_label, device)

    for epoch in range(start_epoch, EPOCHS + 1):
        head.train()
        train_loss_sum = 0.0
        train_count = 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = head(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(x_batch)
            train_count += len(x_batch)
        train_loss = train_loss_sum / max(1, train_count)
        val_metrics = eval_probe(val_loader, val_subjects)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_metrics['loss'], 'val_accuracy': val_metrics['accuracy'], 'val_macro_f1': val_metrics['macro_f1']})
        print(f'[{experiment_name}] epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_metrics["loss"]:.4f} val_acc={val_metrics["accuracy"]:.4f} val_macro_f1={val_metrics["macro_f1"]:.4f}')
        if early_stopper.update(float(val_metrics['macro_f1']), epoch):
            best_state = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
        if SAVE_EVERY > 0 and (epoch % SAVE_EVERY == 0 or epoch == EPOCHS):
            save_checkpoint(latest_checkpoint_path, {'epoch': epoch, 'linear_head_state_dict': head.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_linear_head_state_dict': best_state, 'early_stopper_state_dict': early_stopper.state_dict(), 'history': history, 'label_to_index': label_to_index, 'probe_mode': evaluation_mode, 'encoder_checkpoint_path': encoder_checkpoint_path, 'metadata': checkpoint_metadata(experiment_name, 'probe', {'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'evaluation_mode': evaluation_mode, 'label_fraction': label_fraction, 'input_dim': int(train_x.shape[1]), 'train_windows_used': len(train_split), 'val_windows': len(val_split), 'test_windows': len(test_split), 'save_every': SAVE_EVERY, 'seed': SEED, 'contrastive_experiment_name': contrastive_checkpoint.get('metadata', {}).get('experiment_name')}, windows_root / 'manifest.json', {'checkpoint_role': 'latest', 'checkpoint_epoch': epoch})})
        if early_stopper.should_stop:
            break

    if best_state is None:
        raise RuntimeError(f'[{experiment_name}] probe training finished without a best checkpoint')

    head.load_state_dict(best_state)
    train_metrics = eval_probe(train_loader, train_subjects)
    val_metrics = eval_probe(val_loader, val_subjects)
    test_metrics = eval_probe(test_loader, test_subjects)

    stem = f'probe_{experiment_name}_acc{metric_token(float(test_metrics["accuracy"]))}_macrof1{metric_token(float(test_metrics["macro_f1"]))}'
    checkpoint_path = output_dir / f'{stem}_checkpoint.pt'
    metrics_path = output_dir / f'{stem}_metrics.json'
    confusion_path = output_dir / f'{stem}_confusion_matrix.png'
    per_subject_path = output_dir / f'{stem}_per_subject_accuracy.csv'

    save_checkpoint(checkpoint_path, {'linear_head_state_dict': head.state_dict(), 'probe_mode': evaluation_mode, 'label_fraction': label_fraction, 'encoder_checkpoint_path': encoder_checkpoint_path, 'label_to_index': label_to_index, 'best_epoch': early_stopper.best_epoch, 'metadata': checkpoint_metadata(experiment_name, 'probe', {'batch_size': BATCH_SIZE, 'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'epochs_requested': EPOCHS, 'early_stopping_patience': PATIENCE, 'evaluation_mode': evaluation_mode, 'label_fraction': label_fraction, 'input_dim': int(train_x.shape[1]), 'train_windows_used': len(train_split), 'val_windows': len(val_split), 'test_windows': len(test_split), 'save_every': SAVE_EVERY, 'seed': SEED, 'contrastive_experiment_name': contrastive_checkpoint.get('metadata', {}).get('experiment_name')}, windows_root / 'manifest.json')})
    metrics_payload = {'experiment_name': experiment_name, 'best_epoch': early_stopper.best_epoch, 'probe_mode': evaluation_mode, 'label_fraction': label_fraction, 'train': {'loss': train_metrics['loss'], 'accuracy': train_metrics['accuracy'], 'macro_f1': train_metrics['macro_f1']}, 'val': {'loss': val_metrics['loss'], 'accuracy': val_metrics['accuracy'], 'macro_f1': val_metrics['macro_f1']}, 'test': {'loss': test_metrics['loss'], 'accuracy': test_metrics['accuracy'], 'macro_f1': test_metrics['macro_f1'], 'per_class_f1': test_metrics['per_class_f1'], 'per_subject_accuracy': test_metrics['per_subject_accuracy']}, 'history': history, 'artifacts': {'checkpoint_path': str(checkpoint_path), 'latest_checkpoint_path': str(latest_checkpoint_path), 'metrics_path': str(metrics_path), 'confusion_path': str(confusion_path), 'per_subject_accuracy_path': str(per_subject_path)}}
    metrics_path.write_text(json.dumps(metrics_payload, indent=2))
    plot_confusion_matrix(test_metrics['confusion_matrix'], index_to_label, experiment_name, confusion_path)
    write_subject_accuracy(per_subject_path, test_metrics['per_subject_accuracy'])
    return metrics_payload



def collect_metrics_payloads(*roots: Path, experiment_aliases=None):
    payloads_by_experiment = {}
    experiment_aliases = experiment_aliases or {}
    for root in roots:
        root = Path(root)
        if not root.exists():
            raise FileNotFoundError(f'Metrics root does not exist: {root}')
        for metrics_path in sorted(root.rglob('*_metrics.json')):
            payload = json.loads(metrics_path.read_text())
            original_experiment_name = payload.get('experiment_name')
            experiment_name = experiment_aliases.get(original_experiment_name, original_experiment_name)
            if not experiment_name:
                continue
            if experiment_name in payloads_by_experiment:
                existing = payloads_by_experiment[experiment_name]['metrics_path']
                raise ValueError(f'Duplicate metrics for experiment {experiment_name!r}: {existing} and {metrics_path}')
            payloads_by_experiment[experiment_name] = {
                'payload': payload,
                'metrics_path': str(metrics_path),
                'original_experiment_name': original_experiment_name,
            }
    return payloads_by_experiment



def summarize_runs(payloads_by_experiment, expected_experiments):
    missing = [name for name in expected_experiments if name not in payloads_by_experiment]
    if missing:
        raise FileNotFoundError('Missing expected experiment metrics: ' + ', '.join(missing))
    rows = []
    for experiment_name in expected_experiments:
        payload_entry = payloads_by_experiment[experiment_name]
        payload = payload_entry['payload']
        metadata_config = payload.get('config', {})
        test_metrics = payload.get('test', {})
        probe_mode = payload.get('probe_mode')
        channel_mode = payload.get('channel_mode')
        rows.append({
            'experiment_name': experiment_name,
            'stage': 'probe' if probe_mode else 'supervised',
            'mode': probe_mode or channel_mode,
            'label_fraction': payload.get('label_fraction', metadata_config.get('label_fraction')),
            'test_accuracy': test_metrics.get('accuracy'),
            'test_macro_f1': test_metrics.get('macro_f1'),
            'metrics_path': payload_entry['metrics_path'],
        })
    return rows



def write_comparison_outputs(*, baseline_dir: Path, probe_dir: Path, output_dir: Path, expected_experiments, experiment_aliases=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payloads = collect_metrics_payloads(baseline_dir, probe_dir, experiment_aliases=experiment_aliases)
    rows = summarize_runs(payloads, expected_experiments)

    json_path = output_dir / 'comparison_summary.json'
    markdown_path = output_dir / 'comparison_summary.md'
    json_path.write_text(json.dumps({'results': rows}, indent=2), encoding='utf-8')

    lines = [
        '# Phase 2 Comparison Summary',
        '',
        '| Experiment | Stage | Mode | Label Fraction | Test Accuracy | Test Macro F1 |',
        '| --- | --- | --- | ---: | ---: | ---: |',
    ]
    for row in rows:
        lines.append('| {experiment_name} | {stage} | {mode} | {label_fraction} | {test_accuracy:.4f} | {test_macro_f1:.4f} |'.format(**row))
    markdown_path.write_text('
'.join(lines) + '
', encoding='utf-8')
    return {'json': json_path, 'markdown': markdown_path, 'rows': rows}

In [ ]:
run_results = {}

if RUN_WATCH_FULL:
    run_results['watch_full'] = run_supervised_experiment(
        run_name=WATCH_FULL_RUN_NAME,
        experiment_name=WATCH_FULL_EXPERIMENT,
        channel_mode='watch',
        label_fraction=1.0,
    )
else:
    print('Skipping watch_full supervised run')

if RUN_WATCH_10PCT:
    run_results['watch_10pct'] = run_supervised_experiment(
        run_name=WATCH_10PCT_RUN_NAME,
        experiment_name=WATCH_10PCT_EXPERIMENT,
        channel_mode='watch',
        label_fraction=0.1,
    )
else:
    print('Skipping watch_10pct supervised run')

if RUN_CONTRASTIVE:
    run_results['contrastive'] = run_contrastive_experiment(
        run_name=CONTRASTIVE_RUN_NAME,
        experiment_name=CONTRASTIVE_EXPERIMENT,
    )
    probe_checkpoint_path = run_results['contrastive']['artifacts']['last_checkpoint_path']
    print('Using contrastive checkpoint for probes:', probe_checkpoint_path)
else:
    contrastive_metrics_root = resolve_output_dir('contrastive', CONTRASTIVE_RUN_NAME)
    contrastive_metrics = json.loads(sorted(contrastive_metrics_root.glob('*_metrics.json'))[-1].read_text())
    probe_checkpoint_path = contrastive_metrics['artifacts']['last_checkpoint_path']
    print('Reusing existing contrastive checkpoint:', probe_checkpoint_path)

if RUN_PROBES:
    for evaluation_mode, label_fraction, run_name, experiment_name in PROBE_RUNS:
        run_results[experiment_name] = run_probe_experiment(
            run_name=run_name,
            experiment_name=experiment_name,
            evaluation_mode=evaluation_mode,
            label_fraction=label_fraction,
            encoder_checkpoint_path=probe_checkpoint_path,
        )
else:
    print('Skipping probes')

In [ ]:
if RUN_COMPARE:
    outputs = write_comparison_outputs(
        baseline_dir=Path(OUTPUT_ROOT) / 'supervised',
        probe_dir=Path(OUTPUT_ROOT) / 'probe',
        output_dir=Path(OUTPUT_ROOT) / 'reports' / 'watch_ssl',
        expected_experiments=EXPECTED_EXPERIMENTS,
        experiment_aliases=EXPERIMENT_ALIASES,
    )
    display(pd.DataFrame(outputs['rows']))
    print(json.dumps({'json': str(outputs['json']), 'markdown': str(outputs['markdown'])}, indent=2))
else:
    print('Skipping comparison summary')